In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.8 MB/s eta 0:00:0000:01


In [2]:
import os
import librosa
import numpy as np
import pandas as pd
import torch
from torch_geometric.data import Data, Dataset as PyGDataset
from torch_geometric.loader import DataLoader
from transformers import AutoTokenizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings("ignore")

# 1. Local Kaggle Paths (from your copied clipboard)
MUSICCAPS_CSV = "/kaggle/input/datasets/googleai/musiccaps/musiccaps-public.csv"
AUDIO_DIR = "/kaggle/input/notebooks/osanseviero/musiccaps-explorer/music_data"

print("Loading MusicCaps metadata from local input...")
df = pd.read_csv(MUSICCAPS_CSV)
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

class MusicCapsDataset(PyGDataset):
    def __init__(self, df, audio_dir, tokenizer, max_len=128):
        super().__init__()
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.valid_data = []
        
        print("Matching captions to available audio files...")
        for idx, row in tqdm(df.iterrows(), total=len(df), desc="Verifying files"):
            audio_path = os.path.join(audio_dir, f"{row['ytid']}.wav") 
            if os.path.exists(audio_path):
                self.valid_data.append((row['caption'], audio_path))
                
        print(f"Found {len(self.valid_data)} matching audio-caption pairs!")

    def len(self):
        return len(self.valid_data)

    def get(self, idx):
        caption, audio_path = self.valid_data[idx]
        
        # --- A. Text: Tokenize Caption ---
        encoding = self.tokenizer(
            caption, truncation=True, padding="max_length", max_length=self.max_len, return_tensors="pt"
        )
        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)
        
        # --- B. Audio: Build Segment Graph ---
        try:
            y, sr = librosa.load(audio_path, sr=22050, duration=10.0)
        except:
            y, sr = np.zeros(22050 * 10), 22050

        segment_length = len(y) // 10
        node_features = []
        for i in range(10):
            segment = y[i*segment_length : (i+1)*segment_length]
            mfcc = librosa.feature.mfcc(y=segment, sr=sr, n_mfcc=20) if len(segment) > 0 else np.zeros((20, 10))
            features = np.hstack((np.mean(mfcc, axis=1), np.var(mfcc, axis=1)))
            node_features.append(features)
            
        node_features = np.array(node_features)
        edge_index = [[i, i+1] for i in range(9)] + [[i+1, i] for i in range(9)]
        
        sim_matrix = cosine_similarity(node_features)
        for i in range(10):
            for j in range(i+2, 10):
                if sim_matrix[i, j] > 0.75:
                    edge_index.extend([[i, j], [j, i]])
                    
        x = torch.tensor(node_features, dtype=torch.float)
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        
        data = Data(x=x, edge_index=edge_index)
        data.input_ids = input_ids
        data.attention_mask = attention_mask
        return data

# Build the Dataset
dataset = MusicCapsDataset(df, AUDIO_DIR, tokenizer)

# Split into Train (80%) and Test (20%)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)
print(f"DataLoaders ready! Training batches: {len(train_loader)}")

Loading MusicCaps metadata from local input...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Matching captions to available audio files...


Verifying files:   0%|          | 0/5521 [00:00<?, ?it/s]

Found 32 matching audio-caption pairs!
DataLoaders ready! Training batches: 2


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel
from torch_geometric.nn import SAGEConv, global_mean_pool
from torch.optim import AdamW
from tqdm.auto import tqdm

# --- 1. Dual-Encoder Architecture ---
class ContrastiveDualEncoder(nn.Module):
    def __init__(self, gnn_in=40, gnn_hid=64, text_hid=768, shared_dim=128):
        super().__init__()
        # Text Branch (BERT)
        self.bert = AutoModel.from_pretrained("distilbert-base-uncased")
        self.text_proj = nn.Linear(text_hid, shared_dim) 
        
        # Audio Branch (GraphSAGE)
        self.conv1 = SAGEConv(gnn_in, gnn_hid)
        self.conv2 = SAGEConv(gnn_hid, shared_dim)
        
        # Temperature parameter for Contrastive Loss
        self.tau = nn.Parameter(torch.tensor(0.07)) 

    def forward(self, input_ids, attention_mask, x, edge_index, batch):
        if input_ids.dim() == 1:
            input_ids = input_ids.view(-1, 128)
            attention_mask = attention_mask.view(-1, 128)

        # Encode Text
        H_text = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        t_cls = H_text[:, 0, :]
        t_proj = self.text_proj(t_cls)
        t_norm = F.normalize(t_proj, p=2, dim=1)

        # Encode Audio Graph
        g_x = F.relu(self.conv1(x, edge_index))
        g_x = self.conv2(g_x, edge_index)
        g_raw = global_mean_pool(g_x, batch)
        g_norm = F.normalize(g_raw, p=2, dim=1)

        return t_norm, g_norm, self.tau

# --- 2. InfoNCE Contrastive Loss ---
def info_nce_loss(t_norm, g_norm, tau):
    sim_matrix = torch.matmul(g_norm, t_norm.T) / tau
    labels = torch.arange(t_norm.size(0)).to(t_norm.device)
    loss_g2t = F.cross_entropy(sim_matrix, labels)
    loss_t2g = F.cross_entropy(sim_matrix.T, labels)
    return (loss_g2t + loss_t2g) / 2.0

# --- 3. Training Loop ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ContrastiveDualEncoder().to(device)
optimizer = AdamW(model.parameters(), lr=3e-5)

epochs = 10
print("--- Starting Contrastive Training (Task 4) ---")

for epoch in range(1, epochs + 1):
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}", leave=False)
    
    for batch in pbar:
        batch = batch.to(device)
        optimizer.zero_grad()
        
        t_norm, g_norm, tau = model(batch.input_ids, batch.attention_mask, batch.x, batch.edge_index, batch.batch)
        loss = info_nce_loss(t_norm, g_norm, tau)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix({'NCE_Loss': f"{loss.item():.4f}"})
        
print("Training Complete!")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--- Starting Contrastive Training (Task 4) ---


Epoch 1/10:   0%|          | 0/2 [00:00<?, ?it/s]

...

Epoch 7/10:   0%|          | 0/2 [00:00<?, ?it/s]

Epoch 8/10:   0%|          | 0/2 [00:00<?, ?it/s]

Epoch 9/10:   0%|          | 0/2 [00:00<?, ?it/s]

Epoch 10/10:   0%|          | 0/2 [00:00<?, ?it/s]

Training Complete!


In [5]:
import numpy as np
import torch

print("--- Evaluating Retrieval Metrics ---")
model.eval()
all_t, all_g, captions = [], [], []

# Extract embeddings for the test set
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        t_norm, g_norm, _ = model(batch.input_ids, batch.attention_mask, batch.x, batch.edge_index, batch.batch)
        all_t.append(t_norm.cpu())
        all_g.append(g_norm.cpu())
        
        # FIX: Reshape the flattened PyG tensor back to (Batch_Size, 128)
        input_ids = batch.input_ids
        if input_ids.dim() == 1:
            input_ids = input_ids.view(-1, 128)
            
        # Decode input_ids back to text for our qualitative examples
        for ids in input_ids:
            tokens = tokenizer.convert_ids_to_tokens(ids.tolist())
            clean_tokens = [t for t in tokens if t not in ['[PAD]', '[CLS]', '[SEP]']]
            captions.append(tokenizer.convert_tokens_to_string(clean_tokens))

all_t = torch.cat(all_t, dim=0)
all_g = torch.cat(all_g, dim=0)

# Build the (N x N) similarity matrix
sim_matrix = torch.matmul(all_t, all_g.T).numpy()
N = sim_matrix.shape[0]

# Calculate Recall@K
def get_recall(sim_mat, K_list=[1, 3, 5]):
    recalls = {f"R@{k}": 0 for k in K_list}
    ranked_indices = np.argsort(-sim_mat, axis=1) # Sort descending
    
    for i in range(N):
        for k in K_list:
            if i in ranked_indices[i, :k]: # If true match (i) is in top K
                recalls[f"R@{k}"] += 1
    return {k: round(v / N, 4) for k, v in recalls.items()}

c2a = get_recall(sim_matrix, [1, 3, 5])
a2c = get_recall(sim_matrix.T, [1, 3, 5])

print(f"Test Set Size: {N} audio-caption pairs\n")
print(f"Caption -> Audio Retrieval : {c2a}")
print(f"Audio -> Caption Retrieval : {a2c}")

print("\n==========================================")
print("     Qualitative Retrieval Examples")
print("==========================================")

# Generate top-3 retrieved audio clips for a few text captions
for i in range(min(3, N)):
    print(f"\n[Query Caption {i+1}]: {captions[i]}")
    
    # Get top 3 predicted audio indices for this text caption
    top_audio_indices = np.argsort(-sim_matrix[i, :])[:3]
    
    print("Top 3 Retrieved Audio Tracks:")
    for rank, idx in enumerate(top_audio_indices, 1):
        match_status = "TRUE MATCH" if idx == i else "Mismatch"
        print(f"  {rank}. Track {idx} ({match_status}) - Similarity Score: {sim_matrix[i, idx]:.4f}")

--- Evaluating Retrieval Metrics ---
Test Set Size: 7 audio-caption pairs

Caption -> Audio Retrieval : {'R@1': 0.1429, 'R@3': 0.1429, 'R@5': 0.7143}
Audio -> Caption Retrieval : {'R@1': 0.1429, 'R@3': 0.2857, 'R@5': 0.8571}

     Qualitative Retrieval Examples

[Query Caption 1]: the pop song features a soft female vocal singing over sustained pulsating synth lead, mellow piano melody, sustained synth brass, punchy kick, claps. tinny wide hi hats and high pitched female vocal melody. it sounds emotional, sad, passionate and like something you would hear on a radio.
Top 3 Retrieved Audio Tracks:
  1. Track 6 (Mismatch) - Similarity Score: -0.0026
  2. Track 2 (Mismatch) - Similarity Score: -0.0103
  3. Track 1 (Mismatch) - Similarity Score: -0.0163

[Query Caption 2]: the song is an instrumental. the song is medium tempo, with a horn and string section and a drum section playing to a crescendo, followed by a low frequency thumping rhythm. the song is an instrumental soundtrack for an a